Imports

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

Define Dataset Paths

In [8]:
dataset_path = Path(r"C:\BITS AI ML\Project final\Vegetation_Encroachment_Project\data\raw\vep1\TESELLATED_WITHOUT_AUGMENTATION")

rgb_path = dataset_path / "RGB"
mask_path = dataset_path / "MASK"

rgb_images = sorted(list(rgb_path.glob("*")))
mask_images = sorted(list(mask_path.glob("*")))

print("RGB Images:", len(rgb_images))
print("Mask Images:", len(mask_images))


RGB Images: 532
Mask Images: 532


Train/Validation Split

In [9]:
from sklearn.model_selection import train_test_split

train_images, val_images, train_masks, val_masks = train_test_split(
    rgb_images,
    mask_images,
    test_size=0.2,
    random_state=42
)

print("Train Images:", len(train_images))
print("Validation Images:", len(val_images))

Train Images: 425
Validation Images: 107


Label Remapping

In [11]:
def remap_mask(mask):
    new_mask = np.zeros_like(mask)

    new_mask[mask == 0] = 0
    new_mask[mask == 110] = 1
    new_mask[mask == 149] = 2

    return new_mask

sample_mask = cv2.imread(str(train_masks[0]), cv2.IMREAD_GRAYSCALE)

new_mask = remap_mask(sample_mask)

print("Original values:", np.unique(sample_mask))
print("Remapped values:", np.unique(new_mask))

Original values: [  0 110 149]
Remapped values: [0 1 2]


Augmentations

In [12]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Normalize(),
    ToTensorV2()
])

PyTorch Dataset class

In [17]:
from torch.utils.data import Dataset

class VEPLDataset(Dataset):

    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.imread(str(self.image_paths[idx]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        mask = remap_mask(mask)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"].long()

        return image, mask

Create Dataset Objects

In [18]:
train_dataset = VEPLDataset(
    train_images,
    train_masks,
    transform=train_transform
)

val_dataset = VEPLDataset(
    val_images,
    val_masks,
    transform=val_transform
)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))

Train dataset size: 425
Validation dataset size: 107


Create DataLoaders

In [19]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False
)

print("DataLoaders created successfully!")

DataLoaders created successfully!


Verify Batch Loading

In [20]:
images, masks = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Mask batch shape:", masks.shape)

print("Image dtype:", images.dtype)
print("Mask dtype:", masks.dtype)

print("Unique mask values in batch:", torch.unique(masks))

Image batch shape: torch.Size([8, 3, 256, 256])
Mask batch shape: torch.Size([8, 256, 256])
Image dtype: torch.float32
Mask dtype: torch.int64
Unique mask values in batch: tensor([0, 1, 2])
